# Exact Amplitude Amplification in native FlagQuantum

Complete local computational Notebook workflow: exact amplification, executed seeded sampling with all-success assertion, standard and tuned Grover trajectories and saved trajectory plot.

Reuse the existing native fixed-p=0.07, three-round R/phase composition with explicit clean auxiliary. Local seeded sampling replaces the cloud call, and a saved Matplotlib figure replaces interactive display. Classical trajectory coordinates equal the source rotation products; no native higher-order exact-amplification API or hardware execution claim.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](main.py).

## Native imports for main

In [ ]:
import math
from collections import Counter
from pathlib import Path
from random import Random
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QBit, R, allocate, phase, qfunc

In [ ]:
PHI = math.pi / 2

In [ ]:
THETA = 2 * math.asin(math.sqrt(0.07))

In [ ]:
BETA = 2 * math.acos(math.sin(math.pi / 14) / math.sqrt(0.07))

## main

In [ ]:
@qfunc
def main(target: Output[QBit], aux: Output[QBit]) -> None:
    allocate(target)
    allocate(aux)
    R(THETA, PHI, target)
    R(BETA, PHI, aux)
    # The current native frontend uses static calls; three rounds are explicit.
    phase(target, math.pi)
    phase(target * aux, -math.pi)
    R(-BETA, PHI, aux)
    R(-THETA, PHI, target)
    phase(target, -math.pi)
    phase(aux, -math.pi)
    phase(target * aux, math.pi)
    R(THETA, PHI, target)
    R(BETA, PHI, aux)
    phase(target, math.pi)
    phase(target * aux, -math.pi)
    R(-BETA, PHI, aux)
    R(-THETA, PHI, target)
    phase(target, -math.pi)
    phase(aux, -math.pi)
    phase(target * aux, math.pi)
    R(THETA, PHI, target)
    R(BETA, PHI, aux)
    phase(target, math.pi)
    phase(target * aux, -math.pi)
    R(-BETA, PHI, aux)
    R(-THETA, PHI, target)
    phase(target, -math.pi)
    phase(aux, -math.pi)
    phase(target * aux, math.pi)
    R(THETA, PHI, target)
    R(BETA, PHI, aux)

## sample counts

In [ ]:
def sample_counts(shots: int = 1000, seed: int = 17) -> dict[tuple[int, ...], int]:
    """Sample executed named probabilities locally and check exact success."""
    if type(shots) is not int or shots < 1:
        raise ValueError("shots must be a positive integer")
    program = synth.synthesize(main)
    result = fq.run(program)
    probabilities = dict(program.probabilities(result, registers=("target", "aux")))
    counts = dict(
        Counter(
            Random(seed).choices(
                list(probabilities), weights=list(probabilities.values()), k=shots
            )
        )
    )
    if counts != {(1, 0): shots}:
        raise AssertionError("exact amplification must succeed and clean the auxiliary")
    return counts

## trajectories

In [ ]:
def trajectories() -> (
    tuple[tuple[tuple[float, float], ...], tuple[tuple[float, float], ...]]
):
    """Preserve the Notebook's standard and ancilla-tuned Grover trajectories."""
    theta = math.asin(math.sqrt(0.07))
    rounds = math.ceil(math.pi / (4 * theta) - 0.5)
    tuned = math.pi / (4 * rounds + 2)
    return tuple(
        (math.sin((2 * step + 1) * theta), math.cos((2 * step + 1) * theta))
        for step in range(rounds + 1)
    ), tuple(
        (math.sin((2 * step + 1) * tuned), math.cos((2 * step + 1) * tuned))
        for step in range(rounds + 1)
    )

## plot trajectories

In [ ]:
def plot_trajectories(path: Path) -> None:
    """Save the source's good/bad plane comparison without a cloud dependency."""
    from matplotlib import pyplot as plt

    standard, tuned = trajectories()
    figure, axis = plt.subplots(figsize=(6, 6))
    axis.set_aspect("equal")
    axis.axhline(0, color="gray", linewidth=1)
    axis.axvline(0, color="gray", linewidth=1)
    for states, color, label in (
        (standard, "blue", "amplitude amplification"),
        (tuned, "red", "exact amplitude amplification"),
    ):
        for step, (good, bad) in enumerate(states):
            axis.arrow(
                0,
                0,
                good,
                bad,
                head_width=0.035,
                length_includes_head=True,
                color=color,
                alpha=0.7,
            )
            axis.text(good * 1.08, bad * 1.08, str(step), color=color)
        axis.plot([], [], color=color, label=label)
    axis.set(xlim=(-1, 1.6), ylim=(-1, 1.4), xlabel="|Good>", ylabel="|Bad>")
    axis.legend(loc="upper right")
    figure.tight_layout()
    figure.savefig(path)
    plt.close(figure)

## run example

In [ ]:
def run_example(plot_path: Path | None = None) -> None:
    print(sample_counts())
    if plot_path is not None:
        plot_trajectories(plot_path)
        print(f"Saved Grover trajectories to {plot_path}")

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
plot_path = Path("grover-trajectories.png")

In [ ]:
run_example(plot_path)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(plot_path)))

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states

reference = fq.Circuit(2, dtype=torch.complex128).ry(0, theta=THETA).ry(1, theta=BETA)
for _ in range(3):
    reference.x(1).cz(0, 1).x(1)
    reference.ry(1, theta=-BETA).ry(0, theta=-THETA)
    reference.z(0).z(1).cz(0, 1)
    reference.ry(0, theta=THETA).ry(1, theta=BETA)
compare_states(synth.synthesize(main), reference, ("target", "aux"))
print("Independent full-state comparison: passed")
